In [ ]:
# 04_era1_ramp_characterization.ipynb — Era 1 (2019-2022) intra-day ramp analysis
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip

import pandas as pd
import matplotlib.pyplot as plt

hourly = pd.read_csv("data/study1_hourly.csv")
hourly["datetime"] = pd.to_datetime(hourly["datetime"], format="%d-%m-%Y %H:%M")
hourly["date"] = hourly["datetime"].dt.normalize()
daily = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])

# Restrict to Era 1
hourly = hourly[hourly["date"].dt.year.between(2019, 2022)].copy()
daily_era1 = daily[daily["date"].dt.year.between(2019, 2022)][["date", "share_res_pct"]]

DEMAND_COL = "National Hourly Demand"  # confirm exact column name in your CSV

# --- Hour-to-hour ramp (delta) per day ---
hourly = hourly.sort_values("date").reset_index(drop=True)
hourly["ramp"] = hourly[DEMAND_COL].diff()

# Ramp magnitude = daily max absolute hour-to-hour change
daily_ramp = hourly.groupby(hourly["date"].dt.date)["ramp"].apply(lambda x: x.abs().max())
daily_ramp = daily_ramp.reset_index()
daily_ramp.columns = ["date", "ramp_magnitude"]
daily_ramp["date"] = pd.to_datetime(daily_ramp["date"])

# Ramp frequency = count of hours where |delta| exceeds a threshold (e.g. top 10% of ramps)
threshold = hourly["ramp"].abs().quantile(0.9)
daily_freq = hourly.groupby(hourly["date"].dt.date)["ramp"].apply(lambda x: (x.abs() > threshold).sum())
daily_freq = daily_freq.reset_index()
daily_freq.columns = ["date", "ramp_frequency"]
daily_freq["date"] = pd.to_datetime(daily_freq["date"])

# --- Merge with RES share ---
merged = daily_ramp.merge(daily_freq, on="date").merge(daily_era1, on="date", how="left")

# --- Monthly aggregation for trend clarity ---
merged["month"] = merged["date"].dt.to_period("M")
monthly = merged.groupby("month").agg(
    ramp_magnitude=("ramp_magnitude", "mean"),
    ramp_frequency=("ramp_frequency", "mean"),
    share_res_pct=("share_res_pct", "mean"),
).reset_index()
monthly["month"] = monthly["month"].dt.to_timestamp()

# --- The key chart: ramp magnitude trend vs RES share ---
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly["month"], monthly["ramp_magnitude"], color="tab:blue", label="Ramp magnitude (MW)")
ax1.set_ylabel("Ramp magnitude (MW)", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(monthly["month"], monthly["share_res_pct"], color="tab:orange", label="RES share %")
ax2.set_ylabel("RES share %", color="tab:orange")

plt.title("Era 1 (2019-2022): Intra-day ramp magnitude vs RES share")
plt.show()

# --- Correlation check -
print(monthly[["ramp_magnitude", "ramp_frequency", "share_res_pct"]].corr())

monthly.to_csv("data/era1_ramp_vs_res_share.csv", index=False)